In [ ]:
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle 

In [ ]:
df = pd.read_csv('Churn_Modelling.csv')

In [ ]:
df.head()

In [ ]:
## Preprocess the data 
## Drop irrelevant features 

df.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)

In [ ]:
    df.head()

In [ ]:
## Encode Categorical Variables 

label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])
df.head()

In [ ]:
## One hot encode the Geography column
from sklearn.preprocessing import OneHotEncoder 

OneHotEncoder_geo = OneHotEncoder()
geo_encoder = OneHotEncoder_geo.fit_transform(df[['Geography']])

In [ ]:
OneHotEncoder_geo.get_feature_names_out(['Geography'])

In [ ]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(),columns = OneHotEncoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df.head()

In [ ]:
## combine the one hot encoded columns eith the originasl data 

df = pd.concat([df.drop('Geography',axis=1),geo_encoded_df],axis=1)
df.head()

In [ ]:
## save the encoders and scaler 
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(OneHotEncoder_geo,file)
    

In [ ]:
## Divide the dataset into independednt and dependent variables 

x = df.drop('Exited',axis = 1)
y = df['Exited']

## Split the data into training and testing data
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=0)

## Scale these features 
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


In [ ]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)    

In [ ]:
## ANN Implementation

In [ ]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
## Build our ANN model 
model = Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)), ## Hidden layer connected with input layer
    Dense(32,activation= 'relu'),
    Dense(1,activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)

In [ ]:
## Compile the model 
model.compile(optimizer= opt,loss="binary_crossentropy",metrics = ['accuracy'])

In [ ]:
## setup the ternsorboard 
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir = log_dir,histogram_freq =1)

In [ ]:
## Setup Early Stopping 
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [ ]:
## Train the model 

history = model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs = 100,callbacks = [tensorflow_callback,early_stopping_callback])

In [ ]:
from tensorflow.keras.models import load_model

# Load the legacy HDF5 model
model = load_model('model.h5')

# Save in the Keras format
model.save('model.keras')



In [ ]:
%load_ext tensorboard

In [ ]:





%tensorboard --logdir logs\fit


In [ ]:
## Loas